# Chapter 8: Dynamic Hedging in code

Rebuilds the book's two hedging examples (Figures 8-1 to 8-5) from the underlying prices and deltas, checks every total, and then goes one step further: it **simulates** thousands of random price paths to show *why* the hedge captures the theoretical edge, how rehedging frequency changes the spread of results, and why the breakeven volatility is the implied volatility.
Notes: [[chapters/08-dynamic-hedging]]

Uses numpy, scipy and matplotlib (all in the Anaconda base). Black-Scholes is used as a tool for deltas and values; its formula is covered in a later chapter. Units follow the book: delta in whole numbers (50 means 0.50), interest as simple interest on the number of days, 100 options per position.

In [ ]:
from math import sqrt, exp, log, erf
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

def N(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def bs(S, K, t, r, vol, kind="c"):
    '''Black-Scholes (no dividend stock): returns (value, delta as a decimal). t in years.'''
    d1 = (log(S / K) + (r + vol * vol / 2) * t) / (vol * sqrt(t))
    d2 = d1 - vol * sqrt(t)
    call = S * N(d1) - K * exp(-r * t) * N(d2)
    if kind == "c":
        return call, N(d1)
    return call - S + K * exp(-r * t), N(d1) - 1

def black76_put(F, K, t, r, vol):
    '''Put on a futures contract, stock-type settlement (premium paid up front): (value, delta).'''
    d1 = (log(F / K) + vol * vol / 2 * t) / (vol * sqrt(t))
    d2 = d1 - vol * sqrt(t)
    return exp(-r * t) * (K * N(-d2) - F * N(-d1)), -exp(-r * t) * N(-d1)

def implied_vol(price, value_fn, lo=0.01, hi=1.5):
    '''Bisection: the volatility at which value_fn(vol) equals the price.'''
    for _ in range(80):
        mid = (lo + hi) / 2
        lo, hi = (lo, mid) if value_fn(mid) > price else (mid, hi)
    return mid

## The stock-option example: setup (page 119-121)
Stock 97.70, June 100 call, 10 weeks (70 days), interest 6%, and a "crystal ball" volatility of 37.62%. The call is offered at 5.00. We check the book's theoretical value (5.89), delta (50) and the implied volatility of the 5.00 price (32.40%).

In [ ]:
S0, K, r, vol = 97.70, 100, 0.06, 0.3762
T = 70 / 365                                   # 10 weeks

value, delta = bs(S0, K, T, r, vol)
print(f"theoretical value {value:.2f} (book 5.89), delta {delta * 100:.1f} (book 50)")
print(f"theoretical edge per option: {value:.2f} - 5.00 = {value - 5.00:.2f}  ->  100 options: {100 * (value - 5.00):.2f}  (book +89.00)")

iv = implied_vol(5.00, lambda v: bs(S0, K, T, r, v)[0])
print(f"implied volatility of the 5.00 price: {iv:.2%}  (book 32.40%)")

## Figure 8-1: do the weekly deltas match the model?
The book lists the weekly underlying prices and the June 100 call delta after each week. We recompute the delta with 37.62% volatility and the time left (10 weeks falling by one each week).

In [ ]:
prices = [97.70, 99.50, 92.75, 95.85, 96.20, 102.45, 93.30, 91.15, 95.20, 102.80, 103.85]
book_delta = [50, 54, 35, 43, 43, 62, 28, 17, 27, 72]          # weeks 0 to 9

print("week  price   model delta  book delta")
for k in range(10):
    t = (70 - 7 * k) / 365
    d = bs(prices[k], K, t, r, vol)[1] * 100
    print(f"{k:>4} {prices[k]:>7.2f} {d:>11.1f} {book_delta[k]:>11}")

## Rebuild the adjustments (Figure 8-1)
Each week the total delta position is `100 options x delta - 100 x (contracts short)`. If it is positive we **sell** that many deltas' worth of underlying contracts; if negative we **buy**. The cash is price x contracts (selling brings cash in), and it earns (or costs) interest for the days left to expiration.

In [ ]:
n_opt = 100
short = 50                                   # contracts short in total (original hedge)
rows = []
for k in range(1, 10):
    delta_pos = n_opt * book_delta[k] - 100 * short       # total delta position before adjusting
    contracts = delta_pos / 100                           # + means sell, - means buy
    cash = contracts * prices[k]
    days_left = 70 - 7 * k
    interest = cash * r * days_left / 365
    short += contracts
    rows.append(dict(week=k, price=prices[k], delta=book_delta[k], delta_pos=delta_pos,
                     contracts=contracts, adj_short=short - 50, cash=cash, interest=interest))

print("week  price  delta  total delta  trade        total adj   cash flow  interest")
for x in rows:
    side = "Sell" if x["contracts"] > 0 else "Buy" if x["contracts"] < 0 else "None"
    n = abs(int(x["contracts"]))
    pos = "Short" if x["adj_short"] > 0 else "Long" if x["adj_short"] < 0 else "-"
    print(f"{x['week']:>4} {x['price']:>6.2f} {x['delta']:>5} {x['delta_pos']:>+11.0f}  {side:>4} {n:>3}   {pos:>5} {abs(int(x['adj_short'])):>3} {x['cash']:>+12.2f} {x['interest']:>+9.2f}")

adj_short_end = short - 50
close_cash = -adj_short_end * prices[10]                  # buy back the adjustment shorts at expiration
print(f"week 10: buy {int(adj_short_end)} at {prices[10]} -> {close_cash:+.2f}  (book -2284.70)")

## Figure 8-2: all the profit and loss components
Five pieces: the original hedge (option and stock), the adjustments, interest on the option cost, interest on the stock proceeds, and interest on the adjustments. Then discount back 70 days.

In [ ]:
option_pnl = 100 * (max(prices[10] - K, 0) - 5.00)               # sell the calls at 3.85
stock_pnl = 50 * (97.70 - prices[10])                            # sold at 97.70, bought back at 103.85
adjust_pnl = sum(x["cash"] for x in rows) + close_cash
carry_options = 100 * -5.00 * r * 70 / 365
carry_stock = 50 * 97.70 * r * 70 / 365
interest_adj = sum(x["interest"] for x in rows)

total = option_pnl + stock_pnl + adjust_pnl + carry_options + carry_stock + interest_adj
pv = total / (1 + r * 70 / 365)
predicted = 100 * (5.89 - 5.00)

print(f"Original hedge          {option_pnl + stock_pnl:>9.2f}   (option {option_pnl:.2f}, stock {stock_pnl:.2f})   book -422.50")
print(f"Adjustments             {adjust_pnl:>9.2f}   book +467.55")
print(f"Carry on the options    {carry_options:>9.2f}   book -5.75")
print(f"Carry on the stock      {carry_stock:>9.2f}   book +56.21")
print(f"Interest on adjustments {interest_adj:>9.2f}   book -5.27 in Figure 8-2 (the text says -5.28)")
print(f"Total cash flow         {total:>9.2f}   book +90.24")
print(f"Discounted (70 days)    {pv:>9.2f}   book +89.21")
print(f"Predicted by the model  {predicted:>9.2f}   book +89.00")

labels = ["original\nhedge", "adjustments", "option\ncarry", "stock\ninterest", "interest on\nadjustments"]
parts = [option_pnl + stock_pnl, adjust_pnl, carry_options, carry_stock, interest_adj]
bottoms, run = [], 0
for p in parts:
    bottoms.append(run if p >= 0 else run + p)
    run += p
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.bar(labels, [abs(p) for p in parts], bottom=bottoms, color=["#d62728" if p < 0 else "#2ca02c" for p in parts])
ax.bar(["total cash\nflow"], [total], color="#1f77b4")
ax.bar(["discounted\n(= today)"], [pv], color="#1f77b4")
ax.bar(["model\nprediction"], [predicted], color="#7f7f7f")
for i, p in enumerate(parts):
    ax.text(i, bottoms[i] + abs(p) + 8, f"{p:+.2f}", ha="center", fontsize=9)
for i, v in enumerate([total, pv, predicted], start=len(parts)):
    ax.text(i, v + 8, f"{v:+.2f}", ha="center", fontsize=9)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("profit or loss")
ax.set_title("A big loss on the original hedge, rescued by the adjustments")
plt.tight_layout(); plt.savefig("../images/ch08-pnl-components.png", dpi=130); plt.show()

## Buy low, sell high: what the adjustments actually do
Weekly price and the trade needed to get back to delta neutral. When the price rose the delta rose, so we **sold**; when it fell we **bought**.

In [ ]:
weeks = list(range(11))
fig, (a1, a2) = plt.subplots(2, 1, figsize=(8.5, 6.2), sharex=True)
a1.plot(weeks, prices, marker="o")
a1.axhline(97.70, color="grey", ls=":", lw=1); a1.text(10.1, 97.9, "start 97.70", fontsize=8, ha="right")
a1.set_ylabel("underlying price"); a1.set_title("Weekly prices (Figure 8-1)")
wk = [x["week"] for x in rows]; tr = [x["contracts"] for x in rows]
a2.bar(wk, tr, color=["#d62728" if c > 0 else "#2ca02c" for c in tr])
for w, c in zip(wk, tr):
    if c != 0:
        a2.text(w, c + (1.5 if c > 0 else -4), f"{'sell' if c > 0 else 'buy'} {abs(int(c))}", ha="center", fontsize=8)
a2.axhline(0, color="black", lw=0.8); a2.set_ylim(-42, 54)
a2.set_ylabel("contracts traded"); a2.set_xlabel("week"); a2.set_title("Adjustment needed at each week (red = sell, green = buy)")
plt.tight_layout(); plt.savefig("../images/ch08-hedge-path.png", dpi=130); plt.show()

# Average price paid versus received on the adjustments
sold = sum(c * p for c, p in zip(tr, prices[1:10]) if c > 0); sold_n = sum(c for c in tr if c > 0)
bought = sum(-c * p for c, p in zip(tr, prices[1:10]) if c < 0); bought_n = sum(-c for c in tr if c < 0)
print(f"sold {sold_n:.0f} contracts at an average of {sold / sold_n:.2f}; bought {bought_n:.0f} at an average of {bought / bought_n:.2f} (before the final buy-back)")

## Figure 8-3: the "unhedged amount" is the curvature (gamma)
The dashed line is the call's value if it moved at its initial delta (this is what the opposite stock position offsets); the curve is the true value. The red bar is what the hedge misses after a bigger move, and it is on the same side whether the price rises or falls because the curve bends *up*. Rehedging collects that gap again and again. Panel (a) is the first step (hedge at 97.70, price to 99.50); panel (b) is the second (hedge reset at 99.50 with delta 54, price falls to 92.75). To a good approximation the gap is half the gamma times the move squared.

In [ ]:
def delta_gamma(S_, t_left, h=0.01):
    d_ = bs(S_, K, t_left, r, vol)[1]
    g_ = (bs(S_ + h, K, t_left, r, vol)[1] - bs(S_ - h, K, t_left, r, vol)[1]) / (2 * h)   # change in delta per point
    return d_, g_

# Figure 8-3a: hedge set at 97.70 with 70 days left, price moves to 99.50.
# Figure 8-3b: hedge reset at 99.50 (63 days left), price then moves to 92.75.
# Time is frozen for each comparison so that only the curvature shows (a real week would also cost theta).
steps_3 = [("a", 97.70, 99.50, T), ("b", 99.50, 92.75, 63 / 365)]
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, (label, S_from, S_to, t_left) in zip(axes, steps_3):
    V_from = bs(S_from, K, t_left, r, vol)[0]
    d_from, g_from = delta_gamma(S_from, t_left)
    true = bs(S_to, K, t_left, r, vol)[0]
    line_value = V_from + d_from * (S_to - S_from)
    print(f"8-3{label}: hedge at {S_from} (delta {d_from:.4f}, gamma {g_from:.4f}); price to {S_to}: true value {true:.3f}, "
          f"delta-only {line_value:.3f}, unhedged amount {true - line_value:.3f}, half gamma x move^2 = {0.5 * g_from * (S_to - S_from) ** 2:.3f}")
    Sg = np.linspace(85, 112, 200)
    ax.plot(Sg, [bs(s_, K, t_left, r, vol)[0] for s_ in Sg], lw=2, label="call theoretical value")
    ax.plot(Sg, V_from + d_from * (Sg - S_from), "--", label="delta line (the hedge)")
    ax.plot([S_to, S_to], [line_value, true], color="#d62728", lw=4, label="unhedged amount")
    ax.plot([S_from], [V_from], "ko"); ax.plot([S_to], [true], "ko")
    ax.set_title(f"Figure 8-3{label}: hedge at {S_from}, price moves to {S_to}")
    ax.set_xlabel("underlying price"); ax.set_ylabel("value")
    ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.savefig("../images/ch08-unhedged-amount.png", dpi=130); plt.show()

## Simulation: how often to rehedge?
The book works one price path and says the result matching the theory was "carefully constructed". Here we run the hedge on **20,000 random paths** with the true volatility 37.62% (drift set to the interest rate) and rehedge 1, 10, 70 or 280 times over the 70 days (1 = just the original hedge). For each path the profit is compounded to expiration and discounted back, as in Figure 8-2, per 100 options.

In [ ]:
def bs_call_vec(S, K, t, r, v):
    d1 = (np.log(S / K) + (r + v * v / 2) * t) / (v * np.sqrt(t))
    d2 = d1 - v * np.sqrt(t)
    return S * norm.cdf(d1) - K * np.exp(-r * t) * norm.cdf(d2), norm.cdf(d1)

def hedge_pnl(sig_real, sig_model, steps, n=20000, seed=1, price=5.00):
    '''P&L (per 100 options, discounted to today) of: buy calls at `price`, short delta shares,
    rehedge `steps` times at equal time intervals. Deltas use sig_model; prices move with sig_real.'''
    rng = np.random.default_rng(seed)
    dt = T / steps
    z = rng.standard_normal((n, steps))
    S = S0 * np.exp(np.cumsum((r - sig_real ** 2 / 2) * dt + sig_real * np.sqrt(dt) * z, axis=1))
    S = np.hstack([np.full((n, 1), S0), S])
    d = np.column_stack([bs_call_vec(S[:, k], K, T - k * dt, r, sig_model)[1] for k in range(steps)])
    cash = -price + d[:, 0] * S[:, 0]                     # pay for the call, sell delta shares
    for k in range(1, steps):
        cash = cash * np.exp(r * dt) + (d[:, k] - d[:, k - 1]) * S[:, k]     # more delta -> sell more
    cash = cash * np.exp(r * dt)
    final = np.maximum(S[:, -1] - K, 0) + cash - d[:, -1] * S[:, -1]        # option payoff + cash - buy back
    return 100 * final * np.exp(-r * T)

print(f"theoretical profit: {100 * (bs(S0, K, T, r, vol)[0] - 5.00):.2f}")
results = {}
for steps in (1, 10, 70, 280):
    p = hedge_pnl(vol, vol, steps)
    results[steps] = p
    print(f"rehedges {steps:>3}: average {p.mean():>7.2f}   standard deviation {p.std():>7.2f}   chance of a loss {np.mean(p < 0):.1%}")

plt.figure(figsize=(8.5, 4.8))
for steps, label in ((1, "original hedge only"), (10, "10 times (weekly)"), (70, "70 times (daily)")):
    plt.hist(results[steps], bins=120, range=(-1200, 1400), density=True, histtype="step", lw=1.8, label=label)
plt.axvline(89, color="black", ls=":", lw=1); plt.text(250, plt.ylim()[1] * 0.9, "theory +89", fontsize=9, ha="left")
plt.xlabel("profit on the hedge (per 100 options)"); plt.ylabel("density")
plt.title("Same average, narrower spread as we rehedge more often")
plt.legend(); plt.tight_layout(); plt.savefig("../images/ch08-rehedging-frequency.png", dpi=130); plt.show()

## The race: profit against realized volatility
We hold the model's delta (37.62%) fixed but let the stock move with different **true** volatilities. The book says the hedge breaks even where the true volatility equals the implied volatility of the price paid (32.40%). The line is the theoretical value at the true volatility minus the price paid; the dots are simulated daily-rehedged averages.

In [ ]:
grid = np.linspace(0.18, 0.55, 60)
line = [100 * (bs(S0, K, T, r, g)[0] - 5.00) for g in grid]
sims = [(g, hedge_pnl(g, vol, 70, n=10000, seed=2).mean()) for g in (0.20, 0.25, 0.30, 0.324, 0.35, 0.3762, 0.45, 0.50)]
for g, m in sims:
    print(f"true volatility {g:.1%}: simulated average profit {m:>8.2f}   line {100 * (bs(S0, K, T, r, g)[0] - 5.00):>8.2f}")

plt.figure(figsize=(8, 5))
plt.plot(grid * 100, line, lw=2, label="theoretical value at the true volatility - price paid")
plt.plot([g * 100 for g, _ in sims], [m for _, m in sims], "o", label="simulated average (daily rehedging)")
plt.axhline(0, color="black", lw=0.8)
plt.axvline(iv * 100, color="grey", ls=":"); plt.text(iv * 100 + 0.5, 190, f"breakeven {iv:.2%}\n(implied volatility)", fontsize=9)
plt.axvline(37.62, color="grey", ls=":"); plt.text(37.62 + 0.5, -100, "forecast 37.62%", fontsize=9)
plt.xlabel("true volatility of the stock over the 10 weeks (%)"); plt.ylabel("profit on the hedge (per 100 options)")
plt.title("Higher realized volatility means more adjustments and more profit")
plt.legend(loc="upper left"); plt.tight_layout(); plt.savefig("../images/ch08-hedge-pnl-vs-volatility.png", dpi=130); plt.show()

## Rehedging by delta bands (page 129-130)
Instead of rehedging on a fixed calendar, a trader may rehedge only when the delta position drifts past a limit. Looking at the weekly delta positions from Figure 8-1, which adjustments would each limit have skipped? (This is only the book's first-pass illustration: once you skip a week, later deltas change too.)

In [ ]:
weekly = {x["week"]: x["delta_pos"] for x in rows}
print("week:", " ".join(f"{w:>6}" for w in weekly))
print("delta:", " ".join(f"{v:>+6.0f}" for v in weekly.values()))
for limit in (500, 1000, 1500):
    skipped = [w for w, v in weekly.items() if 0 < abs(v) <= limit]
    print(f"limit {limit:>4} deltas: first-pass adjustments skipped in weeks {skipped} ({len(skipped)} fewer trades)")

## Implied volatility moves while we hold the hedge (page 130-131)
If implied volatility jumps from 32.40% to our forecast 37.62%, the call is worth 5.89 straight away and we can close for the whole edge. If it falls to 30.35%, we show a temporary loss, but the forecast (and so the final profit) has not changed.

In [ ]:
for v in (iv, 0.3762, 0.3035):
    print(f"implied volatility {v:.2%}: call price {bs(S0, K, T, r, v)[0]:.2f}")
print(f"immediate profit if volatility reevaluates to 37.62%: {100 * (bs(S0, K, T, r, 0.3762)[0] - 5.00):+.2f}")
print(f"immediate loss if implied volatility falls to 30.35%: {100 * (bs(S0, K, T, r, 0.3035)[0] - 5.00):+.2f}  (book -35.00)")

## The futures-option example: selling an overpriced put (Figures 8-4 and 8-5)
March 60 put on futures at 61.85, 10 weeks, interest 8%, true volatility 21.48%. The put trades at 1.70 but is worth 1.46 (implied volatility 23.92%). We sell 100 puts and sell 35 futures. Futures have no cash cost, only **variation**: each week we receive (or pay) the price change times the number of futures held, and that cash earns (or costs) interest.

In [ ]:
F0, K2, r2, vol2 = 61.85, 60, 0.08, 0.2148
val, dl = black76_put(F0, K2, T, r2, vol2)
print(f"theoretical value {val:.2f} (book 1.46), delta {dl * 100:.1f} (book -35)")
print(f"implied volatility of the 1.70 price: {implied_vol(1.70, lambda v: black76_put(F0, K2, T, r2, v)[0]):.2%}  (book 23.92%)")

fp = [61.85, 60.83, 62.78, 63.16, 61.68, 59.86, 62.88, 61.50, 62.60, 60.18, 58.61]
put_delta = [-35, -42, -28, -24, -34, -50, -21, -31, -15, -45]
book_variation = [35.70, -81.90, -10.64, 35.52, 61.88, -151.00, 29.98, -34.10, 36.30]

short_f = 35                                            # futures short in total
var_total = interest_total = 0.0
print("week  price  put delta  held short  variation  book      interest  trade")
for k in range(1, 11):
    var = short_f * (fp[k - 1] - fp[k])                 # short futures gain when the price falls
    intr = var * r2 * (70 - 7 * k) / 365
    var_total += var; interest_total += intr
    if k <= 9:
        target = 100 * abs(put_delta[k]) / 100          # short puts: need that many futures short
        trade = target - short_f
        side = "sell" if trade > 0 else "buy"
        flag = "  <- book prints 29.98" if abs(var - book_variation[k - 1]) > 0.02 else ""
        print(f"{k:>4} {fp[k]:>6.2f} {put_delta[k]:>9} {short_f:>10.0f} {var:>+10.2f} {book_variation[k - 1]:>+8.2f} {intr:>+9.2f}  {side} {abs(int(trade))}{flag}")
        short_f = target
    else:
        print(f"{k:>4} {fp[k]:>6.2f} {'':>9} {short_f:>10.0f} {var:>+10.2f} {'':>8} {intr:>+9.2f}  close out {int(short_f)}")

opt_pnl = 100 * (1.70 - max(K2 - fp[10], 0))
fut_pnl = 35 * (fp[0] - fp[10])
adj_pnl = var_total - fut_pnl
carry = 100 * 1.70 * r2 * 70 / 365
total2 = opt_pnl + fut_pnl + adj_pnl + carry + interest_total
print(f"\nOriginal hedge {opt_pnl + fut_pnl:.2f} (option {opt_pnl:.2f}, futures {fut_pnl:.2f})  book +144.40")
print(f"Adjustment P&L {adj_pnl:.2f}  book -122.01")
print(f"Carry on options {carry:.2f}  book +2.61;  interest on variation {interest_total:.2f}  book -0.67")
print(f"Total {total2:.2f}  book +24.33;  discounted {total2 / (1 + r2 * 70 / 365):.2f}  book +23.96;  predicted {100 * (1.70 - 1.46):.2f}")